# Preencher datas e escolher filial

In [11]:
# Data de faturamento (a partir de)
data_inicial_saida_entrega = "02/06/2026"

# Data a ser exibida no excel do protocolo de entrega
data_no_protocolo = "03/06/2026"

# Escolher filial ao trocar chaves da requisição. Altera nome do arquivo excel gerado
# filial = "distribuidora"
filial = "comercio"

# Imports

In [12]:
import requests
import pandas as pd
import os
import json
from dotenv import load_dotenv
from openpyxl import load_workbook
from openpyxl.drawing.image import Image

# Request API

In [13]:
url = "https://app.omie.com.br/api/v1/produtos/nfconsultar/"

# Carrega o .env e escolhe credenciais por filial
load_dotenv()
# Procura variáveis específicas por filial, com fallback para variáveis genéricas
if filial.lower() == "distribuidora":
    app_key = os.getenv("APP_KEY_DISTRIBUIDORA") or os.getenv("app_key") or os.getenv("APP_KEY")
    app_secret = os.getenv("APP_SECRET_DISTRIBUIDORA") or os.getenv("app_secret") or os.getenv("APP_SECRET")
else:
    app_key = os.getenv("APP_KEY_COMERCIO") or os.getenv("app_key") or os.getenv("APP_KEY")
    app_secret = os.getenv("APP_SECRET_COMERCIO") or os.getenv("app_secret") or os.getenv("APP_SECRET")
print(f"Usando credenciais para filial: {filial}")

payload = {
    "call": "ListarNF",
    "app_key": app_key,
    "app_secret": app_secret,
    "param": [{
  "pagina": 1,
  "registros_por_pagina": 100,
  "ordenar_por": "CODIGO",
  "dSaiEntInicial": data_inicial_saida_entrega
}]
}

response = requests.post(url, json=payload)

if response.status_code != 200:
    print(f"Erro na requisição da página 1")
else:
    data = response.json()

# Extrai os dados das notas fiscais do json e armazena em uma lista de tuplas (cnpj_cpf, numero_nf)
nfs = data.get("nfCadastro", [])
todas_nfs = []
for nf in nfs:
    numero_nf = nf.get('ide', {}).get('nNF').lstrip('0')
    cnpj_cpf = nf.get('nfDestInt', {}).get('cnpj_cpf')
    todas_nfs.append((cnpj_cpf, numero_nf))

print(todas_nfs)

Usando credenciais para filial: comercio
[('34.035.902/0002-66', '9615'), ('34.035.902/0004-28', '9616'), ('10.254.717/0005-47', '9617'), ('10.254.717/0004-66', '9618'), ('10.254.717/0008-90', '9619'), ('02.318.956/0003-23', '9620'), ('02.318.956/0012-14', '9621'), ('02.318.956/0013-03', '9622'), ('02.318.956/0008-38', '9623'), ('02.318.956/0006-76', '9624'), ('34.035.902/0006-90', '9625'), ('34.035.902/0005-09', '9626'), ('34.035.902/0001-85', '9627'), ('10.254.717/0002-02', '9628'), ('02.318.956/0007-57', '9629'), ('02.318.956/0014-86', '9630'), ('02.318.956/0004-04', '9631'), ('02.318.956/0020-24', '9632'), ('12.365.759/0002-38', '9633'), ('02.318.956/0015-67', '9634'), ('12.365.759/0003-19', '9635'), ('12.365.759/0001-57', '9636'), ('12.365.759/0004-08', '9637'), ('02.318.956/0020-24', '9638'), ('04.879.925/0001-05', '9639')]


# Algoritmo Nome Fantasia

In [14]:
#importa "clientes_concatenados_atualizada.csv" para variável df_concatenated
df_concatenated = pd.read_csv("clientes_concatenados_atualizada.csv")

#cria cnpj_col para verificar qual coluna de CNPJ existe
cnpj_col = 'cnpj_cpf' if 'cnpj_cpf' in df_concatenated.columns else 'CNPJ'

# substitui os CNPJs de todas_nfs pelos nomes fantasia usando o df_concatenated atualizado
nome_fantasia_col = 'nome_fantasia' if 'nome_fantasia' in df_concatenated.columns else 'Nome Fantasia'
cnpj_to_nome = dict(zip(df_concatenated[cnpj_col], df_concatenated[nome_fantasia_col]))
todas_nfs = [(cnpj_to_nome.get(cnpj, cnpj), num) for cnpj, num in todas_nfs]

print(f"Substituição com df_concatenated atualizado concluída. Total de NFs: {len(todas_nfs)}")
print(todas_nfs)


Substituição com df_concatenated atualizado concluída. Total de NFs: 25
[('ASTOR JK', '9615'), ('COFRE ASTOR', '9616'), ('ICI JK', '9617'), ('ICI VL LOBOS', '9618'), ('ICI ALPHAVILLE', '9619'), ('PIRA ALPHAVILLE', '9620'), ('PRAINHA ITAIM', '9621'), ('PIRA TAMBORE', '9622'), ('PIRA ELDORADO', '9623'), ('ORIGINAL', '9624'), ('ASTOR VL NOVA CONCEICAO', '9625'), ('ASTOR CETENCO', '9626'), ('ASTOR OF', '9627'), ('ICI BELA CINTRA', '9628'), ('PIRA PAULISTA', '9629'), ('PIRA CENTER NORTE', '9630'), ('PIRA MORUMBI', '9631'), ('PIRA CIDADE SP', '9632'), ('LC2', '9633'), ('PIRA VL MARIANA', '9634'), ('LC3', '9635'), ('LC6', '9636'), ('LC4', '9637'), ('PIRA CIDADE SP', '9638'), ('ICI BISTRÔ', '9639')]


# Algoritmo Planilhas

In [15]:
# Configuração de blocos
blocos_por_planilha = 5
linhas_por_bloco = 13
linha_inicio = 7

workbook_index = 1
block_index = 0

wb = load_workbook("protocolo_entrega_alterado.xlsx")
ws = wb.active

for i in range(0, len(todas_nfs), 2):
    # Inicia um novo workbook a cada blocos_por_planilha blocos
    if block_index >= blocos_por_planilha:
        file_name = f"{filial}_protocolo_entrega_parte_{workbook_index}.xlsx"

        celulas = ['A1', 'A14', 'A27', 'A40', 'A53', 'I1', 'I14', 'I27', 'I40', 'I53']
        for celula in celulas:
            nova_img = Image('logo_v2.png')
            nova_img.width = 110
            nova_img.height = 95
            ws.add_image(nova_img, celula)

        wb.save(f"C:\\projetos\\protocolo_nfs_projetos\\output\\{file_name}")

        print(f"Salvou {file_name} com {block_index} blocos")

        workbook_index += 1
        block_index = 0
        wb = load_workbook("protocolo_entrega_alterado.xlsx")
        ws = wb.active

    lote = todas_nfs[i:i+2]
    print(f"Processando lote {block_index+1} da planilha {workbook_index}: {lote}")

    row_base = linha_inicio + block_index * linhas_por_bloco
    print(f"Base row para este bloco: {row_base}")

    if len(lote) == 2:
        cnpj1, num1 = lote[0]
        cnpj2, num2 = lote[1]

        # Escreve CNPJs em pares na mesma linha
        ws.cell(row=row_base, column=3, value=cnpj1)   # Coluna C
        ws.cell(row=row_base, column=11, value=cnpj2)  # Coluna K

        # Escreve data_no_protocolo em pares na linha seguinte
        ws.cell(row=row_base + 1, column=4, value=data_no_protocolo)   # Coluna C
        ws.cell(row=row_base + 1, column=12, value=data_no_protocolo)  # Coluna K

        # Escreve números em pares na linha seguinte
        ws.cell(row=row_base + 3, column=4, value=num1)  # Coluna C
        ws.cell(row=row_base + 3, column=12, value=num2) # Coluna K

        print(f"Bloco {block_index+1} planilha {workbook_index}: row {row_base}/{row_base+3} -> {cnpj1},{cnpj2}/{num1},{num2}")

    elif len(lote) == 1:
        cnpj1, num1 = lote[0]

        ws.cell(row=row_base, column=3, value=cnpj1)   # Coluna C
        ws.cell(row=row_base + 1, column=4, value=data_no_protocolo) # Coluna C
        ws.cell(row=row_base + 3, column=3, value=num1) # Coluna C (seguindo o padrão de par)
        
       

        print(f"Bloco {block_index+1} planilha {workbook_index} (solitário): row {row_base}/{row_base+3} -> {cnpj1}/{num1}")

    else:
        continue

    block_index += 1

# Salva o último workbook em aberto
if block_index > 0:
    file_name = f"{filial}_protocolo_entrega_parte_{workbook_index}.xlsx"
    
    celulas = ['A1', 'A14', 'A27', 'A40', 'A53', 'I1', 'I14', 'I27', 'I40', 'I53']
    for celula in celulas:
        nova_img = Image('logo_v2.png')
        nova_img.width = 110
        nova_img.height = 95
        ws.add_image(nova_img, celula)

    wb.save(f"C:\\projetos\\protocolo_nfs_projetos\\output\\{file_name}")
    print(f"Salvou {file_name} com {block_index} blocos")

Processando lote 1 da planilha 1: [('ASTOR JK', '9615'), ('COFRE ASTOR', '9616')]
Base row para este bloco: 7
Bloco 1 planilha 1: row 7/10 -> ASTOR JK,COFRE ASTOR/9615,9616
Processando lote 2 da planilha 1: [('ICI JK', '9617'), ('ICI VL LOBOS', '9618')]
Base row para este bloco: 20
Bloco 2 planilha 1: row 20/23 -> ICI JK,ICI VL LOBOS/9617,9618
Processando lote 3 da planilha 1: [('ICI ALPHAVILLE', '9619'), ('PIRA ALPHAVILLE', '9620')]
Base row para este bloco: 33
Bloco 3 planilha 1: row 33/36 -> ICI ALPHAVILLE,PIRA ALPHAVILLE/9619,9620
Processando lote 4 da planilha 1: [('PRAINHA ITAIM', '9621'), ('PIRA TAMBORE', '9622')]
Base row para este bloco: 46
Bloco 4 planilha 1: row 46/49 -> PRAINHA ITAIM,PIRA TAMBORE/9621,9622
Processando lote 5 da planilha 1: [('PIRA ELDORADO', '9623'), ('ORIGINAL', '9624')]
Base row para este bloco: 59
Bloco 5 planilha 1: row 59/62 -> PIRA ELDORADO,ORIGINAL/9623,9624
Salvou comercio_protocolo_entrega_parte_1.xlsx com 5 blocos
Processando lote 1 da planilha 2: